In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import init
from kaggledatahandler import KaggleDataHandler
from audiopreprocessing import SoundDS
from dnn_model.dnn import AudioClassifier


### Creating dataset (Training & Test)

In [2]:
KDHandler = KaggleDataHandler()
datasets_filepath_organized, y_datasets_filepath_organized = KDHandler.create_set()
print("Created the following filepaths with correspondong label\n:", datasets_filepath_organized.keys())


Created the following filepaths with correspondong label
: dict_keys(['fold1', 'fold2', 'fold3', 'fold4', 'fold5', 'fold6', 'fold7', 'fold8', 'fold9', 'fold10'])


In [3]:

preprocessed_datasets = {}
audiopreprocessor = SoundDS()

for fold in datasets_filepath_organized:
    new_prepro_fold = []
    for i, file in enumerate(datasets_filepath_organized[fold]):
        class_ID = y_datasets_filepath_organized[fold][i]
        filepath = file
        spectrgram, class_id = audiopreprocessor.__getitem__(filepath, class_ID)
        preprocessed_wav = (spectrgram, class_id)
        new_prepro_fold.append(preprocessed_wav)
    print("Finished", fold)
    preprocessed_datasets[fold] = new_prepro_fold

Finished fold1
Finished fold2
Finished fold3
Finished fold4
Finished fold5
Finished fold6
Finished fold7
Finished fold8
Finished fold9
Finished fold10


### Testing loop

In [8]:

def test(model, val_dl):
  correct_prediction = 0
  total_prediction = 0

  # Disable gradient updates
  with torch.no_grad():
    for data in val_dl:
      # Get the input features and target labels, and put them on the GPU
      inputs, labels = data[0].to(device), data[1].to(device)

      # Normalize the inputs
      inputs_m, inputs_s = inputs.mean(), inputs.std()
      inputs = (inputs - inputs_m) / inputs_s

      # Get predictions
      outputs = model(inputs)

      # Get the predicted class with the highest score
      _, prediction = torch.max(outputs,1)
      # Count of predictions that matched the target label
      correct_prediction += (prediction == labels).sum().item()
      total_prediction += prediction.shape[0]
    
  acc = correct_prediction/total_prediction
  print(f'Accuracy: {acc:.2f}, Total items: {total_prediction}\n')
  return acc


### Training loop

In [9]:
def training(model, train_dl, num_epochs, device):
  # Loss Function, Optimizer and Scheduler
  criterion = nn.CrossEntropyLoss()
  optimizer = torch.optim.Adam(model.parameters(),lr=0.001)
  scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.001,
                                                steps_per_epoch=int(len(train_dl)),
                                                epochs=num_epochs,
                                                anneal_strategy='linear')
  
  losses = []
  accuracies = []

  # Repeat for each epoch
  for epoch in range(num_epochs):
    running_loss = 0.0
    correct_prediction = 0
    total_prediction = 0

    # Repeat for each batch in the training set
    for i, data in enumerate(train_dl):
        # Get the input features and target labels, and put them on the GPU
        inputs, labels = data[0].to(device), data[1].to(device)

        # Normalize the inputs
        inputs_m, inputs_s = inputs.mean(), inputs.std()
        inputs = (inputs - inputs_m) / inputs_s

        # Zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()

        # Keep stats for Loss and Accuracy
        running_loss += loss.item()

        # Get the predicted class with the highest score
        _, prediction = torch.max(outputs,1)
        # Count of predictions that matched the target label
        correct_prediction += (prediction == labels).sum().item()
        total_prediction += prediction.shape[0]

        #if i % 10 == 0:    # print every 10 mini-batches
        #    print('[%d, %5d] loss: %.3f' % (epoch + 1, i + 1, running_loss / 10))
    
    # Print stats at the end of the epoch
    num_batches = len(train_dl)
    avg_loss = running_loss / num_batches
    acc = correct_prediction/total_prediction
    losses.append(avg_loss)
    accuracies.append(acc)

    print(f'Epoch: {epoch}, Loss: {avg_loss:.2f}, Accuracy: {acc:.2f}')

  print('Finished Training\n')
  return losses, accuracies

### Training & Test

In [ ]:
number_of_test_folds = 2
sets = KDHandler.create_splits(number_of_test_folds)
models = []
num_epochs = 20

training_losses_all = []
training_accuracies_all = []
test_accuracies_all = []


for i, combination_set in enumerate(sets):

    # Create the model and put it on the GPU if available
    new_model = AudioClassifier()
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    new_model = new_model.to(device)
    # Check that it is on Cuda
    next(new_model.parameters()).device

    print(f"##### Set {i} ####\n")

    print(f" - Starting training for set #{i}\n")
    for fold in combination_set[1]:
        print(f"Training on fold #{fold}\n")
        train_dl = torch.utils.data.DataLoader(preprocessed_datasets[fold], batch_size=8, shuffle=False)
        losses, accuracies = training(new_model, train_dl, num_epochs, device)
        training_losses_all.append(sum(losses)/len(losses))
        training_accuracies_all.append(sum(accuracies)/len(accuracies))
    
    print(f"Starting testing for set #{i}\n")   
    for fold in combination_set[0]:
        print(f"Testing on fold #{fold}")
        test_dl = torch.utils.data.DataLoader(preprocessed_datasets[fold], batch_size=8, shuffle=False)  
        acc = test(new_model, test_dl)
        test_accuracies_all.append(acc)

    models.append(new_model)

print("Training accuracies for all folds:", training_accuracies_all)
print("Test accuracies for all folds:", test_accuracies_all)
    

##### Set 0 ####

 - Starting training for set #0

Training on fold #fold3

Epoch: 0, Loss: 2.22, Accuracy: 0.16
Epoch: 1, Loss: 1.94, Accuracy: 0.35
Epoch: 2, Loss: 1.69, Accuracy: 0.46
Epoch: 3, Loss: 1.43, Accuracy: 0.57
Epoch: 4, Loss: 1.19, Accuracy: 0.65
Epoch: 5, Loss: 1.02, Accuracy: 0.70
Epoch: 6, Loss: 0.91, Accuracy: 0.73
Epoch: 7, Loss: 0.83, Accuracy: 0.76
Epoch: 8, Loss: 0.77, Accuracy: 0.79
Epoch: 9, Loss: 0.73, Accuracy: 0.81
Finished Training

Training on fold #fold4

Epoch: 0, Loss: 1.87, Accuracy: 0.37
Epoch: 1, Loss: 1.55, Accuracy: 0.46
Epoch: 2, Loss: 1.34, Accuracy: 0.53
Epoch: 3, Loss: 1.17, Accuracy: 0.60
Epoch: 4, Loss: 1.02, Accuracy: 0.68
Epoch: 5, Loss: 0.91, Accuracy: 0.73
Epoch: 6, Loss: 0.82, Accuracy: 0.76
Epoch: 7, Loss: 0.76, Accuracy: 0.79
Epoch: 8, Loss: 0.70, Accuracy: 0.80
Epoch: 9, Loss: 0.67, Accuracy: 0.82
Finished Training

Training on fold #fold5

Epoch: 0, Loss: 1.82, Accuracy: 0.39
Epoch: 1, Loss: 1.36, Accuracy: 0.55
Epoch: 2, Loss: 1.11, 